In [29]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

In [40]:
BASE_DIR = "/Users/shivengarisa/Documents/lung_caner_prediction/Combined_Lung_Cancer_Dataset"
TRAIN_DIR = os.path.join(BASE_DIR, "Train")
TEST_DIR  = os.path.join(BASE_DIR, "Test")

BACKBONE = "b3"     # "b1" (faster) or "b3" (more accurate)
IMG_SIZE = 300 if BACKBONE == "b3" else 240
BATCH = 32
EPOCHS_STAGE1 = 15
EPOCHS_STAGE2 = 10
UNFREEZE_TOP_K = 30
AUTOTUNE = tf.data.AUTOTUNE
NUM_CLASSES = 3
SEED = 42
MODEL_DIR = "model"
os.makedirs(MODEL_DIR, exist_ok=True)
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

# Attempt to enable op determinism (helps reproducibility; may slow down)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    # if not available on your TF build, ok to proceed
    pass

# Set seeds (Python / NumPy / TF)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

In [31]:
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")

# ---------- helpers ----------
def collect_images_simple(root_dir):
    rows = []
    if not os.path.isdir(root_dir):
        return pd.DataFrame(columns=["filename", "class"])
    for cls in sorted(os.listdir(root_dir)):
        cls_dir = os.path.join(root_dir, cls)
        if not os.path.isdir(cls_dir):
            continue
        for dp, _, files in os.walk(cls_dir):
            for fname in files:
                if fname.lower().endswith(IMG_EXTS):
                    rows.append((os.path.join(dp, fname), cls))
    return pd.DataFrame(rows, columns=["filename", "class"]).reset_index(drop=True)

# ---------- dataset / metadata ----------
train_all_df = collect_images_simple(TRAIN_DIR)
test_df = collect_images_simple(TEST_DIR)
print(f"Found train files: {len(train_all_df)}  test files: {len(test_df)}")

# stratified split
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
idx = np.arange(len(train_all_df))
y = train_all_df["class"].values
train_idx, val_idx = next(sss.split(idx, y))
train_df = train_all_df.iloc[train_idx].reset_index(drop=True)
val_df   = train_all_df.iloc[val_idx].reset_index(drop=True)

# label mapping
classes = sorted(train_all_df["class"].unique())
class_to_idx = {c:i for i,c in enumerate(classes)}
print("classes:", classes)
train_df["label"] = train_df["class"].map(class_to_idx)
val_df["label"]   = val_df["class"].map(class_to_idx)
test_df["label"]  = test_df["class"].map(class_to_idx)

# class/sample weights (normalized mean=1)
sklearn_cw = compute_class_weight(class_weight="balanced",
                                  classes=np.unique(train_df["label"]),
                                  y=train_df["label"])
class_weights = {int(i): float(w) for i,w in enumerate(sklearn_cw)}
cw_vals = np.array([class_weights[i] for i in range(NUM_CLASSES)], dtype=np.float32)
cw_vals = cw_vals / cw_vals.mean()
print("normalized cw_vals:", cw_vals)

train_df["weight"] = train_df["label"].map(lambda l: float(cw_vals[l]))
val_df["weight"]   = val_df["label"].map(lambda l: float(cw_vals[l]))
test_df["weight"]  = test_df["label"].map(lambda l: float(cw_vals[l]))

Found train files: 14184  test files: 3682
classes: ['Benign', 'Malignant', 'Normal']
normalized cw_vals: [1.302944   0.80622554 0.8908307 ]


In [32]:
def scan_for_corrupts(df, n=50):
    sample = df["filename"].sample(min(len(df), n), random_state=SEED)
    for p in sample:
        try:
            with Image.open(p) as im:
                im.verify()
        except Exception as e:
            print("Corrupt or unreadable image:", p, e)

In [33]:
def _load_and_preprocess_pil(path):
    # path: bytes tensor -> convert to string and load via PIL in numpy_function
    path_str = path.decode("utf-8") if isinstance(path, (bytes, bytearray)) else str(path)
    try:
        with Image.open(path_str) as im:
            im = im.convert("RGB")
            im = im.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
            arr = np.asarray(im, dtype=np.float32)  # shape (H,W,3), values 0..255
    except Exception as e:
        # return white image on error (keeps pipeline stable)
        print("⚠️ PIL load failed, returning blank image for:", path_str, e)
        arr = np.ones((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32) * 255.0
    return arr

In [34]:
def _decode_img(path, label, weight=None):
    # Use tf.numpy_function to call PIL loader (safe on mac)
    img = tf.numpy_function(_load_and_preprocess_pil, [path], tf.float32)
    img.set_shape((IMG_SIZE, IMG_SIZE, 3))
    # Keep values as float32 in 0..255 (we'll call EfficientNet preprocess inside model)
    label = tf.cast(label, tf.int32)
    if weight is not None:
        weight = tf.cast(weight, tf.float32)
        return img, label, weight
    return img, label

In [35]:
augmentation_pipeline = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.06),
    layers.RandomZoom(0.06),
    layers.RandomTranslation(0.04, 0.04),
], name="augmentation")

In [36]:

def make_dataset(df, batch=BATCH, shuffle=False, augment_flag=False, include_weights=False):
    if include_weights:
        ds = tf.data.Dataset.from_tensor_slices((df["filename"].values, df["label"].values, df["weight"].values))
        ds = ds.map(_decode_img, num_parallel_calls=AUTOTUNE)
    else:
        ds = tf.data.Dataset.from_tensor_slices((df["filename"].values, df["label"].values))
        ds = ds.map(_decode_img, num_parallel_calls=AUTOTUNE)

    if shuffle:
        # smaller/controlled buffer speeds up first epoch and avoids too long fill times
        ds = ds.shuffle(buffer_size=min(len(df), 2048), seed=SEED, reshuffle_each_iteration=True)

    if augment_flag:
        if include_weights:
            ds = ds.map(lambda x,y,w: (augmentation_pipeline(x, training=True), y, w), num_parallel_calls=AUTOTUNE)
        else:
            ds = ds.map(lambda x,y: (augmentation_pipeline(x, training=True), y), num_parallel_calls=AUTOTUNE)

    ds = ds.batch(batch, drop_remainder=False).prefetch(AUTOTUNE)
    return ds

# create datasets (dataset yields x: float32 0..255, y: int scalar, w: float scalar)
train_ds = make_dataset(train_df, batch=BATCH, shuffle=True, augment_flag=True, include_weights=True)
val_ds   = make_dataset(val_df,   batch=BATCH, shuffle=False, augment_flag=False, include_weights=True)
test_ds  = make_dataset(test_df,  batch=BATCH, shuffle=False, augment_flag=False, include_weights=True)

# optionally ignore irrecoverable errors in pipeline (keeps training running)
train_ds = train_ds.apply(tf.data.experimental.ignore_errors())
val_ds   = val_ds.apply(tf.data.experimental.ignore_errors())
test_ds  = test_ds.apply(tf.data.experimental.ignore_errors())

In [37]:
def build_effnet_backbone(backbone='b3', img_size=IMG_SIZE, num_classes=NUM_CLASSES,
                          head_units=512, head_dropout=0.35):
    inp = keras.Input(shape=(img_size, img_size, 3), name='input_image', dtype=tf.float32)
    # EfficientNet preprocess expects float in 0..255
    x = layers.Lambda(lambda t: tf.keras.applications.efficientnet.preprocess_input(t),
                      name="effnet_preprocess", dtype='float32')(inp)

    if backbone.lower() == 'b1':
        Base = tf.keras.applications.EfficientNetB1
        weights_file = "efficientnetb1_notop.h5"
        weights_url = "https://storage.googleapis.com/keras-applications/efficientnetb1_notop.h5"
    elif backbone.lower() == 'b3':
        Base = tf.keras.applications.EfficientNetB3
        weights_file = "efficientnetb3_notop.h5"
        weights_url = "https://storage.googleapis.com/keras-applications/efficientnetb3_notop.h5"
    else:
        raise ValueError("backbone must be 'b1' or 'b3'")

    base = Base(include_top=False, weights=None, input_shape=(img_size, img_size, 3), pooling='avg')
    weights_path = tf.keras.utils.get_file(weights_file, origin=weights_url, cache_subdir="models")
    base.load_weights(weights_path)
    base.trainable = False

    feats = base(x, training=False)

    h = layers.BatchNormalization()(feats)
    h = layers.Dense(head_units, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-5))(h)
    h = layers.BatchNormalization()(h)
    h = layers.Dropout(head_dropout)(h)
    out = layers.Dense(num_classes, dtype='float32')(h)  # logits (no softmax)

    model = keras.Model(inputs=inp, outputs=out, name=f"EffNet_{backbone}_head")
    return model, base

In [38]:
model, base = build_effnet_backbone(backbone=BACKBONE, img_size=IMG_SIZE, num_classes=NUM_CLASSES,
                                   head_units=512, head_dropout=0.35)

# ---------- compile / train ----------
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
opt_stage1 = tf.keras.optimizers.Adam(3e-4)
opt_stage2 = tf.keras.optimizers.Adam(1e-5)

model.compile(optimizer=opt_stage1, loss=loss_fn, metrics=['accuracy'])

callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ModelCheckpoint(os.path.join(MODEL_DIR, 'best_effnet_head.keras'), monitor='val_loss', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
]

print("MODEL SUMMARY")
model.summary()

MODEL SUMMARY


Model: "EffNet_b3_head"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)        │ (None, 300, 300, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ effnet_preprocess (Lambda)      │ (None, 300, 300, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb3 (Functional)     │ (None, 1536)           │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 1536)           │         6,144 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 512)            │       786,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,580,210 (44.17 MB)

 Trainable params: 792,579 (3.02 MB)

 Non-trainable params: 10,787,631 (41.15 MB)

In [ ]:
print("\nSTAGE 1: train head only")
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE1,
    callbacks=callbacks,
    verbose=1
)


STAGE 1: train head only
Epoch 1/15
⚠️ PIL load failed, returning blank image for: /Users/shivengarisa/Documents/lung_caner_prediction/Combined_Lung_Cancer_Dataset/Train/Benign/dataset1_contour_crop (17).jpg [Errno 60] Operation timed out
⚠️ PIL load failed, returning blank image for: /Users/shivengarisa/Documents/lung_caner_prediction/Combined_Lung_Cancer_Dataset/Train/Benign/dataset1_gaussian_blur (67).jpg [Errno 60] Operation timed out
⚠️ PIL load failed, returning blank image for: /Users/shivengarisa/Documents/lung_caner_prediction/Combined_Lung_Cancer_Dataset/Train/Malignant/dataset1_colorjitter (11).jpg [Errno 60] Operation timed out
⚠️ PIL load failed, returning blank image for: /Users/shivengarisa/Documents/lung_caner_prediction/Combined_Lung_Cancer_Dataset/Train/Benign/dataset1_gaussian_blur (106).jpg [Errno 60] Operation timed out
⚠️ PIL load failed, returning blank image for: /Users/shivengarisa/Documents/lung_caner_prediction/Combined_Lung_Cancer_Dataset/Train/Benign/datas

/Users/shivengarisa/image_classification/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


⚠️ PIL load failed, returning blank image for: /Users/shivengarisa/Documents/lung_caner_prediction/Combined_Lung_Cancer_Dataset/Train/Benign/dataset1_colorjitter (51).jpg [Errno 60] Operation timed out
⚠️ PIL load failed, returning blank image for: /Users/shivengarisa/Documents/lung_caner_prediction/Combined_Lung_Cancer_Dataset/Train/Malignant/dataset1_contour_crop (16).jpg [Errno 60] Operation timed out
⚠️ PIL load failed, returning blank image for: /Users/shivengarisa/Documents/lung_caner_prediction/Combined_Lung_Cancer_Dataset/Train/Malignant/dataset1_gaussian_blur (78).jpg [Errno 60] Operation timed out
⚠️ PIL load failed, returning blank image for: /Users/shivengarisa/Documents/lung_caner_prediction/Combined_Lung_Cancer_Dataset/Train/Malignant/dataset1_auto_contrast (12).jpg [Errno 60] Operation timed out
⚠️ PIL load failed, returning blank image for: /Users/shivengarisa/Documents/lung_caner_prediction/Combined_Lung_Cancer_Dataset/Train/Benign/dataset1_vertical_flip (38).jpg [Errn

2025-11-21 19:46:16.617825: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:453] ShuffleDatasetV3:1434: Filling up shuffle buffer (this may take a while): 1300 of 2048


⚠️ PIL load failed, returning blank image for: /Users/shivengarisa/Documents/lung_caner_prediction/Combined_Lung_Cancer_Dataset/Train/Malignant/dataset1_contour_crop (60).jpg [Errno 60] Operation timed out
⚠️ PIL load failed, returning blank image for: /Users/shivengarisa/Documents/lung_caner_prediction/Combined_Lung_Cancer_Dataset/Train/Malignant/dataset1_gaussian_blur (42).jpg [Errno 60] Operation timed out
⚠️ PIL load failed, returning blank image for: /Users/shivengarisa/Documents/lung_caner_prediction/Combined_Lung_Cancer_Dataset/Train/Malignant/dataset1_vertical_flip (19).jpg [Errno 60] Operation timed out
⚠️ PIL load failed, returning blank image for: /Users/shivengarisa/Documents/lung_caner_prediction/Combined_Lung_Cancer_Dataset/Train/Malignant/dataset1_contour_crop (76).jpg [Errno 60] Operation timed out
⚠️ PIL load failed, returning blank image for: /Users/shivengarisa/Documents/lung_caner_prediction/Combined_Lung_Cancer_Dataset/Train/Benign/dataset1_rotate (4).jpg [Errno 60

2025-11-21 19:46:21.727604: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:483] Shuffle buffer filled.


⚠️ PIL load failed, returning blank image for: /Users/shivengarisa/Documents/lung_caner_prediction/Combined_Lung_Cancer_Dataset/Train/Benign/dataset1_colorjitter (114).jpg [Errno 60] Operation timed out
  1/355 ━━━━━━━━━━━━━━━━━━━━ 1:43:13 17s/step - accuracy: 0.7812 - loss: 0.3670⚠️ PIL load failed, returning blank image for: /Users/shivengarisa/Documents/lung_caner_prediction/Combined_Lung_Cancer_Dataset/Train/Benign/dataset1_gaussian_blur (33).jpg [Errno 60] Operation timed out
⚠️ PIL load failed, returning blank image for: /Users/shivengarisa/Documents/lung_caner_prediction/Combined_Lung_Cancer_Dataset/Train/Benign/dataset1_vertical_flip (7).jpg [Errno 60] Operation timed out
⚠️ PIL load failed, returning blank image for: /Users/shivengarisa/Documents/lung_caner_prediction/Combined_Lung_Cancer_Dataset/Train/Benign/dataset1_gaussian_blur (108).jpg [Errno 60] Operation timed out
  2/355 ━━━━━━━━━━━━━━━━━━━━ 11:26 2s/step - accuracy: 0.8125 - loss: 0.3333   ⚠️ PIL load failed, returni

In [ ]:

print("\nEvaluation after Stage 1:")
test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print("Stage1 test_acc:", test_acc, "loss:", test_loss)

In [ ]:
# ---------- STAGE 2: unfreeze top layers ----------
if UNFREEZE_TOP_K > 0:
    for layer in base.layers:
        layer.trainable = False
    # make sure UNFREEZE_TOP_K doesn't exceed base size
    n_unfreeze = min(UNFREEZE_TOP_K, len(base.layers))
    for layer in base.layers[-n_unfreeze:]:
        layer.trainable = True

# freeze all BatchNorm layers (keep inference behavior)
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=opt_stage2, loss=loss_fn, metrics=['accuracy'])

In [ ]:
print("\nSTAGE 2: fine-tuning (top layers)")
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE2,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# save final model
final_path = os.path.join(MODEL_DIR, "lung_cancer_model_best.keras")
model.save(final_path, include_optimizer=False)
print("Saved final model:", final_path)

print("\nFinal evaluation:")
test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print("Final test accuracy:", test_acc, " loss:", test_loss)

In [ ]:
# ---------- metrics / confusion ----------
y_true = []
y_pred = []
for batch in test_ds:
    # dataset yields (x, y, w)
    if len(batch) == 3:
        x_batch, y_batch, _ = batch
    else:
        x_batch, y_batch = batch
    preds = model.predict(x_batch, verbose=0)
    y_true.append(y_batch.numpy())
    y_pred.append(np.argmax(preds, axis=1))
y_true = np.concatenate(y_true, axis=0)
y_pred = np.concatenate(y_pred, axis=0)

from sklearn.metrics import confusion_matrix, classification_report
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.show()

print("Classification report:")
print(classification_report(y_true, y_pred, target_names=classes))